In [ ]:
import os
import pickle
import pandas as pd
from pathlib import Path
from tqdm import tqdm

🔍 总共寻找到 100 个 pickle 文件，开始智能合并解析...


Parsing Files: 100%|██████████| 100/100 [00:00<00:00, 99888.16it/s]


⚠️ 未提取到任何有效数据，请检查 ROOT_DIRS 路径和解析规则。


In [ ]:
# ================= 配置区 =================
ROOT_DIRS = ["../result/archive-103"] 
OUTPUT_FILE_LOC = "./result_101/experiment_results_PowerBI.csv"
TARGET_BUDGETS = {6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0}
# ==========================================

def process_experiment_data(root_paths):
    data_records = []
    pckl_files = []
    
    for r in root_paths:
        path_obj = Path(r)
        if path_obj.exists():
            pckl_files.extend(list(path_obj.rglob("*.pckl")))
        else:
            print(f"⚠️ 警告: 找不到目录 {r}")
            
    print(f"🔍 总共寻找到 {len(pckl_files)} 个 pickle 文件，开始解析...")
    
    for pckl_file in tqdm(pckl_files, desc="Parsing Files"):
        try:
            seed_str = pckl_file.parent.name
            num_str = pckl_file.parent.parent.name
            task_name = pckl_file.parent.parent.parent.name
            
            if task_name == 'caltech' and num_str == '767':
                continue
            
            filename = pckl_file.stem
            parts = filename.split('-')
            
            record = {
                'Task': task_name,
                'Ground_Size': int(num_str),
                'Seed': int(seed_str),
                'Algorithm': None,
                'Strategy': None,
                'UB': None,
                'D': None,
                'Budget': None,
                'Alpha': None,
                'Model': None,
            }

            try:
                budget = float(parts[-3])
                alpha = float(parts[-2])
                model = parts[-1]
                
                if not any(abs(budget - t) < 1e-4 for t in TARGET_BUDGETS):
                    continue
                
                record['Budget'] = budget
                record['Alpha'] = alpha
                record['Model'] = model

                if parts[0] == 'EfficientBFS':
                    # ========= 修改区域 =========
                    # 将 EfficientBFS-traditional-ub0 记录为 simple
                    if parts[1] == 'traditional' and parts[2] == 'ub0':
                        record['Algorithm'] = 'simple'
                    else:
                        record['Algorithm'] = 'EfficientBFS'
                    # ============================
                    
                    record['Strategy'] = parts[1]
                    record['UB'] = parts[2]
                    record['D'] = parts[3]
                elif parts[0] in ['BFSTC', 'Efficient']:
                    record['Algorithm'] = parts[0]
                    record['UB'] = parts[1]
                    record['D'] = parts[2]
                else:
                    continue
                    
            except (ValueError, IndexError):
                continue

            with open(pckl_file, 'rb') as f:
                res = pickle.load(f)
                
            raw_time = res.get('time', None)
            raw_tle = res.get('TLE', False)
            
            if raw_time is not None and raw_time >= 5000:
                raw_time = 5000
                raw_tle = True
                
            record.update({
                'Objective_f(S)': res.get('f(S)', None),
                'Cost_c(S)': res.get('c(S)', None),
                'Time_s': raw_time,
                'Node_Count': res.get('node_count', None),
                'Open_List_Count': res.get('open_list_count', None),
                'TLE': raw_tle,
                'Solution_Set_Size': len(res.get('S', [])) if 'S' in res else 0 
            })
            
            data_records.append(record)
            
        except Exception as e:
            print(f"❌ 解析出错 {pckl_file.name}: {e}")

    df = pd.DataFrame(data_records)
    if not df.empty:
        df.sort_values(by=['Task', 'Algorithm', 'UB', 'Budget', 'Seed'], inplace=True)
        Path(OUTPUT_FILE_LOC).parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(OUTPUT_FILE_LOC, index=False, encoding='utf-8-sig')
        print(f"\n✅ 处理完毕！提取 {len(df)} 条记录。")
        print(f"💾 保存至: {OUTPUT_FILE_LOC}")
    else:
        print("\n⚠️ 未找到匹配数据，请检查路径及 TARGET_BUDGETS。")
        
    return df

if __name__ == "__main__":
    df_results = process_experiment_data(ROOT_DIRS)
    if not df_results.empty:
        print(df_results.head())

🔍 总共寻找到 180 个 pickle 文件，开始解析...


Parsing Files: 100%|██████████| 180/180 [00:00<00:00, 1916.66it/s]


✅ 处理完毕！提取 170 条记录。
💾 保存至: ./result_101/experiment_results_PowerBI_v103.csv
    Task  Ground_Size  Seed Algorithm Strategy   UB  D  Budget  Alpha  \
6  adult          111     0     BFSTC     None  ub0  d     6.0   0.95   
7  adult          111     0     BFSTC     None  ub0  d     7.0   0.95   
8  adult          111     0     BFSTC     None  ub0  d     8.0   0.95   
9  adult          111     0     BFSTC     None  ub0  d     9.0   0.95   
0  adult          111     0     BFSTC     None  ub0  d    10.0   0.95   

                         Model  Objective_f(S)  Cost_c(S)  Time_s  Node_Count  \
6  AdultIncomeFeatureSelection        7.206555   5.836918  5000.0        1568   
7  AdultIncomeFeatureSelection        7.206555   6.999571  5000.0         100   
8  AdultIncomeFeatureSelection        8.120308   7.606156  5000.0          79   
9  AdultIncomeFeatureSelection        8.270787   8.931538  5000.0          72   
0  AdultIncomeFeatureSelection        8.655823   9.305434  5000.0          71   